## 개요

ReAct(Reasoning + Acting)은 프린스턴 대학교 학자들이 구글 연구원들과 협력하여 개발한 프롬프트 엔지니어링 전략입니다.  

이 전략은 LLMs(대형 언어 모델)이 인간이 현실에서 행동하는 방식처럼 먼저 문제를 어떻게 풀지 `생각`하고, `행동`을 하는 것을 흉내내는 전략입니다.  

이때 ReAct는 다수의 `외부 도구`를 사용할 수 있음을 전제하며, `외부 도구`의 결과로부터 `생각`과 `행동`을 다시 진행할 수 있습니다.  

이러한 상호작용은 모델이 질문을 하거나 작업을 수행하여 더 많은 정보를 얻고 상황을 더 잘 이해할 수 있도록 합니다. 예를 들어, 여러 단계를 거쳐야만 해결할 수 있는 문제가 주어져도, ReAct는 외부 도구를 호출하는 여러 검색 작업을 수행하여 문제를 해결합니다.  

LLM이 `생각`과 `행동`을 번갈아 가며 수행하도록 함으로써, ReAct는 LLM을 실제 행동하는 `에이전트`로 변환하여 인간과 유사한 방식으로 작업을 완료할 수 있게 합니다.

ReACT 에이전트는  

- `생각`, `행동`, `행동 입력`, `검색 결과` 를 반복하여 문제를 해결합니다.  
- 문제를 해결하기 위한 모든 `검색 결과`가 취합되면 마지막으로 `최종 답`을 작성합니다.  
- `생각`: 현 시점에서 어떤 도구에 어떤 검색어를 넣어야 할 지 고민하는 단계입니다. 문제를 풀기 위해서 풀이 과정을 기재하므로 CoT 프롬프팅 전략에 해당됩니다.
- `행동`: 실제로 어떤 도구를 사용할 지 결정하는 단계입니다.  
- `행동 입력`: 선택한 도구에 넣을 검색어를 결정하는 단계입니다.  
- `검색 결과`: 도구로부터 검색 결과를 얻습니다. 이 결과를 통해서 다음 `생각`이 전개됩니다.

In [ ]:
!pip install -U "langchain>=1.0.0" "langchain-openai>=0.2.0" "langchain-community>=0.3.0" \
               "langchain-text-splitters>=0.2.0" "tiktoken>=0.7.0" "chromadb>=0.5.5" \
               "pymupdf>=1.24.0" "gradio>=4.44.0" "requests>=2.31.0"

In [ ]:
import os
import re
import io
import sys
import json
import requests
from typing import Dict, List, Optional, Tuple
from contextlib import redirect_stdout

# LangChain 1.0 계열 임포트
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_core.tools import create_retriever_tool
from langchain_core.prompts import PromptTemplate

import gradio as gr

/tmp/ipykernel_864/408261864.py:13: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [ ]:
# 키는 Colab 왼쪽 🔑(보안 비밀)에 OPENAI_API_KEY 이름으로 저장하세요
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## 데이터 로드

In [ ]:
urls = [
   "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_japan_2024.pdf",
   "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_usa_2024.pdf"
]

# 각 파일 다운로드
for url in urls:
   filename = url.split("/")[-1]  # URL에서 파일명 추출
   response = requests.get(url)

   with open(filename, "wb") as f:
       f.write(response.content)
   print(f"{filename} 다운로드 완료")

ict_japan_2024.pdf 다운로드 완료
ict_usa_2024.pdf 다운로드 완료


Github에 저장된 일본과 미국의 ICT 정책 보고서 PDF를 로컬 환경으로 가져오는 코드입니다.  

urls 리스트는 Github의 raw 콘텐츠 URL을 담고 있습니다. 각 URL은 2024년 발행된 일본과 미국의 ICT 정책 보고서를 가리킵니다.  

반복문은 각 URL의 파일을 차례로 다운로드합니다. split() 메서드로 URL의 마지막 부분에서 파일명을 추출하고, requests로 원격 파일의 내용을 받아옵니다. 받아온 내용은 바이너리 쓰기 모드로 로컬에 저장되며, 저장이 끝날 때마다 완료 메시지가 출력됩니다. 이 파일들은 벡터 데이터베이스 구축을 위한 원본 자료로 사용될 예정입니다.

## 임베딩 및 벡터 DB 구현

In [ ]:
# 임베딩 설정
embd = OpenAIEmbeddings()

PDF 파일은 아래 파일을 인터넷에 검색하면 나오는데 그걸 업로드하세요

In [ ]:
def create_pdf_retriever(
    pdf_path: str,  # PDF 파일 경로
    persist_directory: str,  # 벡터 스토어 저장 경로
    embedding_model: OpenAIEmbeddings,  # OpenAIEmbeddings 임베딩 모델
    chunk_size: int = 512,  # 청크 크기 기본값 512
    chunk_overlap: int = 0  # 청크 오버랩 크기 기본값 0
) -> Chroma.as_retriever:

    # PDF 파일로드
    loader = PyMuPDFLoader(pdf_path)
    data = loader.load()

    # 청킹
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    doc_splits = text_splitter.split_documents(data)

    # 벡터 스토어로 적재
    vectorstore = Chroma.from_documents(
        persist_directory=persist_directory,
        documents=doc_splits,
        embedding=embedding_model,
    )

    return vectorstore.as_retriever()

create_pdf_retriever 함수는 PDF 문서를 벡터 데이터베이스로 변환하고 이를 검색할 수 있는 retriever를 생성하는 함수입니다. 이 함수는 PDF 파일 경로, 벡터 저장소 경로, OpenAI 임베딩 모델을 필수 입력으로 받으며, 청크 크기와 오버랩 크기는 선택적 매개변수로 기본값이 각각 512와 0으로 설정되어 있습니다.  

함수 내부적으로 PDF를 처리하는 과정은 세 단계로 이루어집니다. 먼저 PyMuPDF 라이브러리로 PDF를 읽어들입니다. 그 다음 RecursiveCharacterTextSplitter를 사용해 텍스트를 청크 단위로 분할합니다. 이는 긴 문서를 다수의 문서로 분할하기 위함입니다. 분할된 다수의 문서들은 OpenAI의 임베딩 모델을 통해 Chroma 벡터 데이터베이스에 저장됩니다. 마지막으로 create_pdf_retriever 함수는 생성된 벡터 데이터베이스의 검색 인터페이스(retriever)를 반환합니다. 이 retriever는 자연어 질의에 대해 관련성 높은 PDF 내용을 검색할 수 있는 검색기입니다.  

이제 위 함수를 이용하여 일본 ICT 정책에 대해서 검색하는 검색기와 미국 ICT 정책에 대해서 검색하는 검색기를 각각 만들어봅시다.

## 각 벡터 DB를 도구로 등록

In [ ]:
# 일본 ICT 정책 데이터베이스 생성
retriever_japan = create_pdf_retriever(
    pdf_path="ict_japan_2024.pdf",
    persist_directory="db_ict_policy_japan_2024",
    embedding_model=embd
)

# 미국 ICT 정책 데이터베이스 생성
retriever_usa = create_pdf_retriever(
    pdf_path="ict_usa_2024.pdf",
    persist_directory="db_ict_policy_usa_2024",
    embedding_model=embd
)

create_pdf_retriever 함수를 사용해 일본과 미국의 ICT 정책 문서를 각각의 벡터 데이터베이스 검색기로 생성합니다. persist_directory에 "db_ict_policy_japan_2024"와 "db_ict_policy_usa_2024"라는 서로 다른 경로를 지정함으로써, 두 문서의 벡터 데이터베이스가 서로 영향을 주지 않고 독립적으로 저장되고 관리됩니다. 만약 같은 경로를 사용했다면 두 번째 문서 처리 시 첫 번째 문서의 데이터가 덮어써지거나 혼합될 수 있기 때문에, 이렇게 경로를 분리하는 것이 중요합니다.  

일본 ICT 정책의 경우 "ict_japan_2024.pdf" 파일을 읽어 "db_ict_policy_japan_2024" 디렉토리에 벡터 데이터베이스를 생성합니다. 마찬가지로 미국 ICT 정책은 "ict_usa_2024.pdf" 파일을 "db_ict_policy_usa_2024" 디렉토리에 저장합니다.  

이렇게 두 개의 검색기를 만들었습니다. 이제 이렇게 만들어진 두 개의 retriever 객체를 create_retriever_tool 함수를 통해 ReAct 에이전트가 사용할 수 있는 검색 도구로 변환해봅시다.

In [ ]:
jp_engine = create_retriever_tool(
    retriever=retriever_japan,
    name="japan_ict",
    description="일본의 ICT 시장동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요.",
)

usa_engine = create_retriever_tool(
    retriever=retriever_usa,
    name="usa_ict",
    description="미국의 ICT 시장동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요.",
)

In [ ]:
tools = [jp_engine, usa_engine]

In [ ]:
tools

[StructuredTool(name='japan_ict', description='일본의 ICT 시장동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x7d77a5a12980>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x7d77a5a12c00>),
 StructuredTool(name='usa_ict', description='미국의 ICT 시장동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x7d77a5a13060>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x7d77a5a13920>)]

In [ ]:
tool_map: Dict[str, object] = {t.name: t for t in tools}

create_retriever_tool 함수는 retriever 객체를 ReACT 에이전트가 활용할 수 있는 형태의 도구로 변환해주는 함수입니다. 앞서 생성한 retriever_japan과 retriever_usa 객체를 바탕으로 도구의 이름(name)과 설명(description)을 추가하여 ReAct 에이전트가 활용할 수 있는 형태로 만듭니다.  

description에는 각 검색기의 상세한 용도를 작성해야 합니다. 예를 들어 일본 ICT 검색기의 경우 "일본의 ICT 시장동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요"라는 설명을 작성했습니다. 여기서 주의할 점은 각 도구에 대한 description을 매우 상세하게 작성해야 한다는 점입니다. 이후 ReAct 에이전트가 동작할 때, 에이전트는 여기서 적힌 설명을 보고 사용자의 질문에 따라서 도구를 선택합니다. 따라서 description에 도구에 대한 설명이 제대로 적혀있지 않다면, 에이전트는 상황에 맞는 도구 선택을 제대로 할 수 없으므로 주의합니다.

생성된 jp_engine과 usa_engine은 tools 리스트에 담았습니다. 이 tools 리스트는 뒤의 코드에서 ReAct 에이전트에게 전달됩니다. 에이전트는 이 도구들의 상세한 description을 통해 각 질문에 가장 적합한 도구를 선택하고 효과적인 검색을 수행할 수 있게 됩니다.

## ReAct 에이전트 프롬프트 작성

In [ ]:
react_template = '''다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:

{tools}

다음 형식을 사용하세요:

Question: 답변해야 하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요
Action: 취해야 할 행동, [{tool_names}] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
... (이 Thought/Action/Action Input/Observation의 과정이 N번 반복될 수 있습니다)
Thought: 이제 최종 답변을 알겠습니다
Final Answer: 원래 입력된 질문에 대한 최종 답변 (한글로 작성하십시오.)

## 추가적인 주의사항
- 반드시 [Thought -> Action -> Action Input -> Observation] 순서를 준수하십시오. 항상 Action 전에는 Thought가 먼저 나와야 합니다.
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한 번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것도 고려하십시오.
- 정보가 취합되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question: {input}
{agent_scratchpad}'''

In [ ]:
prompt = PromptTemplate.from_template(react_template)

완성된 프롬프트
```
다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:

japan_ict: 일본의 ICT 시장동향 정보를 제공합니다. 일본 ICT와 관련된 질문은 해당 도구를 사용하세요.
usa_ict: 미국의 ICT 시장동향 정보를 제공합니다. 미국 ICT와 관련된 질문은 해당 도구를 사용하세요.

다음 형식을 사용하세요:

Question: 답변해야 하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요
Action: 취해야 할 행동, [japan_ict, usa_ict] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
... (이 Thought/Action/Action Input/Observation의 과정이 N번 반복될 수 있습니다)
Thought: 이제 최종 답변을 알겠습니다
Final Answer: 원래 입력된 질문에 대한 최종 답변

## 추가적인 주의사항
- 반드시 [Thought -> Action -> Action Input format] 이 사이클의 순서를 준수하십시오. 항상 Action 전에는 Thought가 먼저 나와야 합니다.
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한 번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것은 중요합니다.
- 정보가 취합되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question: {사용자가 가장 처음에 하는 질문}
Thought:
```

In [ ]:
def _format_tools_for_prompt(ts: List[object]) -> Tuple[str, str]:
    lines, names = [], []
    for t in ts:
        names.append(t.name)
        desc = getattr(t, "description", "")
        lines.append(f"{t.name}: {desc}")
    return "\n".join(lines), ", ".join(names)

def _render_prompt(user_input: str, scratchpad: str) -> str:
    # tool이름과 tool 설명을 조립하는 단계
    tools_str, tool_names = _format_tools_for_prompt(tools)

    # 프롬프트에다가 tool 이름, tool 설명, 사용자의 질문을 입력하는 단계. 즉, 초기 프롬프트 최종 완성 단계
    return prompt.format(
        tools=tools_str,
        tool_names=tool_names,
        input=user_input,
        agent_scratchpad=scratchpad, # 이전 단계의 사이클은 다음 LLM 호출에서 누적되어서 프롬프트에 부착되어야 됩니다.
    )

`_format_tools_for_prompt` 함수

이 함수는 도구 목록을 프롬프트에 삽입할 수 있는 문자열 형태로 변환하는 용도로 사용됩니다.

예를 들어 `ICT_industry_search`라는 도구가 있고 설명이 `"This PDF provides information about trends..."`라면, 이 함수를 거치면 첫 번째 반환값으로 `"ICT_industry_search: This PDF provides information about trends..."`라는 문자열이 나오고, 두 번째 반환값으로 `"ICT_industry_search"`라는 이름만 나옵니다.

도구가 여러 개라면 첫 번째 반환값은 줄바꿈(`\\n`)으로 연결되고, 두 번째 반환값은 쉼표(`, `)로 연결됩니다. 이렇게 두 가지를 분리해서 반환하는 이유는 프롬프트 템플릿에서 `{tools}` 자리에는 도구 설명 전체가, `{tool_names}` 자리에는 이름 목록만 들어가야 하기 때문입니다.

`_render_prompt` 함수

이 함수는 LLM에 실제로 전달할 최종 프롬프트 문자열을 조립하는 용도로 사용됩니다.

예를 들어 사용자가 `"세계 인공지능 시장규모를 년도 별로 설명해줘"`라고 질문했다면, `user_input`에 이 문장이 들어가고 `prompt.format()`을 통해 템플릿의 `{input}` 자리에 채워집니다.

`scratchpad` 파라미터는 ReAct 루프가 반복될 때마다 이전 단계의 Thought, Action, Observation 기록이 누적된 문자열입니다. 예를 들어 첫 번째 반복에서 LLM이 도구를 호출하고 결과를 받았다면, 두 번째 반복에서는 그 기록이 `scratchpad`에 담겨서 `{agent_scratchpad}` 자리에 삽입됩니다. 이렇게 해야 LLM이 이전에 무슨 행동을 했고 어떤 결과를 받았는지 맥락을 유지하면서 다음 단계를 결정할 수 있습니다.

## 모델 설정

In [ ]:
llm = ChatOpenAI(model="gpt-4.1", temperature=0)

## 프롬프트 파싱

In [ ]:
# =========================
# ReAct 파서 및 실행 루프
# =========================
ACTION_RE = re.compile(r"^Action\s*:\s*(?P<tool>.+?)\s*$", re.MULTILINE)
ACTION_INPUT_RE = re.compile(r"^Action Input\s*:\s*(?P<input>.+?)\s*$", re.MULTILINE)
FINAL_ANSWER_RE = re.compile(r"Final Answer\s*:\s*(?P<final>[\s\S]+)$", re.IGNORECASE)

`ACTION_RE`, `ACTION_INPUT_RE`, `FINAL_ANSWER_RE`는 LLM 응답에서 특정 패턴을 추출하는 용도로 사용됩니다.

예를 들어 LLM이 아래와 같이 응답했다면:

```
Thought: ICT 산업 동향을 검색해야겠다
Action: ICT_industry_search
Action Input: 인공지능 시장규모
```

`ACTION_RE`는 `Action:` 뒤의 `ICT_industry_search`를 추출하고, `ACTION_INPUT_RE`는 `Action Input:` 뒤의 `인공지능 시장규모`를 추출합니다. `FINAL_ANSWER_RE`는 LLM이 `Final Answer:`로 최종 답변을 제시했을 때 그 내용을 추출합니다.

In [ ]:
# 파싱하는 함수
def _parse_action_and_input(text: str) -> Tuple[Optional[str], Optional[str]]:
    # 위에서 만든 도구 이름과 도구에 대한 입력을 파싱하기 위한 정규 표현식을 내부적으로 사용.
    m_act = ACTION_RE.search(text)
    m_in = ACTION_INPUT_RE.search(text)
    if m_act and m_in:
        tool = m_act.group("tool").strip()
        action_input = m_in.group("input").strip()
        action_input_first_line = action_input.splitlines()[0].strip()
        return tool, action_input_first_line

    m_final = FINAL_ANSWER_RE.search(text)
    if m_final:
        return "__FINAL__", m_final.group("final").strip()

    return None, None

이 함수는 LLM 응답 텍스트를 파싱해서 다음에 취할 행동을 판별하는 용도로 사용됩니다.

먼저 `Action:`과 `Action Input:`이 존재하는지 확인합니다. 두 항목이 모두 존재하면, `Action:` 뒤의 도구 이름과 `Action Input:` 뒤의 입력값을 추출하여 `(tool_name, action_input)` 형태로 반환합니다. 이때 `Action Input`에 여러 줄이 포함될 수 있으므로 첫 줄만 사용합니다.

`Action`과 `Action Input`이 없고 `Final Answer:`가 포함되어 있다면, `("__FINAL__", 최종답변텍스트)`를 반환합니다. `"__FINAL__"`은 루프를 종료하라는 신호로 사용됩니다.

둘 다 찾지 못하면 `(None, None)`을 반환하여 파싱 실패를 알립니다.

In [ ]:
# 실제 도구 실행 결과를 프롬프트에 붙이기 위해서 전처리하는 단계.
def _observation_to_text(observation_obj) -> str:
    if isinstance(observation_obj, list):
        # Document 리스트일 수 있음
        def doc_to_str(d):
            try:
                meta = getattr(d, "metadata", {}) or {}
                src = meta.get("source") or meta.get("file_path") or ""
                txt = getattr(d, "page_content", "")
                if len(txt) > 500:
                    txt = txt[:500] + "..."
                return f"[source={src}] {txt}"
            except Exception:
                return str(d)
        return "\n".join(doc_to_str(d) for d in observation_obj[:5])
    return str(observation_obj)

이 함수는 도구 실행 결과를 문자열로 변환하는 용도로 사용됩니다.

예를 들어 `create_retriever_tool`로 만든 검색 도구는 `Document` 객체 리스트를 반환합니다. 이 함수는 각 `Document`에서 `page_content`와 `metadata`의 `source` 정보를 추출해서 `"[source=파일경로] 문서내용..."`와 같은 형태로 변환합니다. 문서 내용이 500자를 넘으면 잘라내고 `...`를 붙입니다. 최대 5개 문서까지만 처리하는데, 너무 많은 내용이 `scratchpad`에 쌓이면 컨텍스트 윈도우를 낭비하기 때문입니다.

## 에이전트 실행 함수

In [ ]:
STOP_SEQ = ["\nObservation:"]  # LLM이 Observation 내용을 쓰기 직전에 출력이 끊기게 함

# ReACT 에이전트를 사이클들 돌게하는 단계
# LLM은 Observation:에서 생성을 멈춰야 한다. 왜냐면 그 이후는 LLM이 생성을 하는 것이 아니라 검색 결과를 붙여야 하기 때문입니다.
def run_react(user_input: str, max_iters: int = 8) -> Dict[str, str]:
    scratchpad = ""

    # 사이클을 돌되 최대 사이클 횟수는 정해져있음.
    for _ in range(max_iters):
        # 프롬프트 구성 (이전 사이클 누적도 포함)
        rendered = _render_prompt(user_input, scratchpad)
        # 실제 LLM 호출
        resp = llm.invoke(rendered, stop=STOP_SEQ)
        text = resp.content if hasattr(resp, "content") else str(resp)

        # LLM 응답으로 Action과 Action Input을 파싱
        tool, action_input = _parse_action_and_input(text)
        if tool is None:
            hint = "\n[파싱안내] 형식을 엄격히 따르세요. 반드시 'Action:'와 'Action Input:'을 한 줄씩 제공하십시오.\n"
            scratchpad += f"{text}\n{hint}"
            continue

        if tool == "__FINAL__":
            final_answer = action_input
            return {"output": final_answer, "log": scratchpad + "\n" + text}

        if tool not in tool_map:
            observation = f"[에러] 존재하지 않는 도구입니다: {tool}"
            scratchpad += f"{text}\nObservation: {observation}\n"
            continue

        try:
            observation_obj = tool_map[tool].invoke(action_input)
            observation = _observation_to_text(observation_obj)
            scratchpad += f"{text}\nObservation: {observation}\n"
        except Exception as e:
            scratchpad += f"{text}\nObservation: [도구실행오류] {e}\n"

    return {
        "output": "반복 한도를 초과했습니다. 질문을 더 구체화해 주세요.",
        "log": scratchpad,
    }

이 함수는 ReAct 에이전트의 메인 실행 루프로, Thought → Action → Action Input → Observation 사이클을 반복하는 용도로 사용됩니다.

`STOP_SEQ = ["\\nObservation:"]`는 LLM이 `Observation:` 이후 내용을 직접 생성하지 못하도록 하기 위한 설정이며, `llm.invoke(rendered, stop=STOP_SEQ)` 호출 시 적용됩니다. 따라서 `Observation` 내용은 반드시 코드에서 실제 도구 실행 결과로 채워집니다.

예를 들어 사용자가 `"세계 인공지능 시장규모를 년도 별로 설명해줘"`라고 질문하면 다음과 같은 흐름으로 진행됩니다:

1. `_render_prompt`로 프롬프트를 조립하고 `llm.invoke(rendered, stop=STOP_SEQ)`로 LLM을 호출합니다.
2. `_parse_action_and_input`으로 응답을 파싱합니다.
3. 파싱 실패 시 힌트 메시지를 `scratchpad`에 추가하고 다시 시도합니다.
4. `"__FINAL__"`이면 최종 답변을 반환하고 루프를 종료합니다.
5. 도구 이름이 `tool_map`에 없으면 에러 메시지를 `Observation:` 형태로 `scratchpad`에 추가합니다.
6. 도구가 유효하면 `tool_map[tool].invoke(action_input)`으로 실제 도구를 실행하고, 결과를 `_observation_to_text`로 변환하여 `Observation:`으로 `scratchpad`에 추가합니다.

`scratchpad`는 매 반복마다 누적되어 다음 LLM 호출 시 `{agent_scratchpad}` 자리에 포함됩니다. 이를 통해 LLM은 이전 단계에서 어떤 도구를 호출했고 어떤 결과를 받았는지 맥락을 유지한 채 다음 행동을 결정합니다.

`max_iters`는 무한 루프 방지용으로, 기본값 8회 안에 `Final Answer`가 나오지 않으면 반복 한도 초과 메시지를 반환합니다.

In [ ]:
# =========================
# 멀티 쿼리
# =========================
try:
    q = "미국과 일본의 ICT 주요 정책의 공통점과 차이점을 설명해주고 미국의 블록체인 스타트업 지원 제도를 정리해 주세요."
    out = run_react(q, max_iters=10)
    print("최종 답변:", out["output"])
    print("\n=== 실행 로그 (Thought / Action / Action Input / Observation) ===\n")
    print(out["log"])
except Exception as e:
    print(f"멀티 쿼리 중 오류: {e}")

최종 답변: 미국과 일본의 ICT 주요 정책의 공통점과 차이점, 그리고 미국의 블록체인 스타트업 지원 제도는 다음과 같이 정리할 수 있습니다.

### 1. 미국과 일본 ICT 주요 정책의 공통점

- **첨단 기술 육성**: 양국 모두 반도체, 인공지능(AI), 양자컴퓨팅, 6G 등 첨단 ICT 기술의 연구개발과 산업 육성에 국가적 역량을 집중하고 있습니다.
- **디지털 인프라 강화**: 데이터센터, 클라우드, 디지털 산업 기반 확충 등 디지털 인프라 강화에 정책적 지원을 아끼지 않습니다.
- **사이버보안 강화**: 사이버보안 위협에 대응하기 위한 정책과 기술 개발을 적극적으로 추진하고 있습니다.
- **국제 협력 확대**: 미국과 일본 모두 기술 경쟁력 강화를 위해 국가 간 협력(특히 양자컴퓨팅, 반도체 등) 및 글로벌 파트너십을 강화하고 있습니다.

### 2. 미국과 일본 ICT 주요 정책의 차이점

- **정책 추진 방식**:  
  - 미국은 민간 주도의 혁신과 대규모 정부 투자(예: CHIPS Act 등)를 병행하며, 빅테크 기업 중심의 생태계가 특징입니다.  
  - 일본은 정부 주도의 전략적 산업 육성(예: 산업경쟁력 강화법 개정, 전략 분야 집중 지원)과 규제 완화, 국내 생산 촉진에 더 무게를 두고 있습니다.
- **특화 분야**:  
  - 미국은 우주 클라우드 컴퓨팅, 드론 배송, 의료 AI 등 다양한 신산업 분야에서 선도적 역할을 하고 있습니다.  
  - 일본은 Web3(블록체인 기반 차세대 인터넷), 자체 소프트웨어 개발, 행정 AI 도입 등에서 정책적 특색을 보입니다.
- **시장 구조**:  
  - 미국은 글로벌 ICT 시장을 주도하는 빅테크 기업(구글, 애플, 마이크로소프트 등)이 중심입니다.  
  - 일본은 대기업과 정부의 협력, 그리고 국내 산업 보호 및 육성에 초점을 맞추고 있습니다.

### 3. 미국의 블록체인 스타트업 지원 제도

- **정부 및 민간 협력**: 미국은 블록체인 및 Web3 스타트업을 위한 다

현재 질문은 "미국과 일본의 ICT 주요 정책의 공통점과 차이점"이었으며, 사용 가능한 도구는 retriever_japan과 retriever_usa입니다. 이는 하나의 질문이지만 두 국가의 정보를 필요로 하는 복합적인 성격을 띠고 있습니다.  

ReAct 에이전트가 시작되자 에이전트는 질문의 의도에 따라서 적절한 도구를 배정하기 위해서 Thought(생각 과정)을 거칩니다. 생각 과정을 통해 두 국가의 정책을 비교하기 위해서는 각각의 정보가 필요하다고 판단하여, 하나의 질문을 두 단계로 나누어 접근합니다. 먼저 Action 단계에서 retriever_usa 도구를 선택하고 Action Input 단계에서 "미국의 ICT 주요 정책"이라는 검색어를 사용합니다.  

미국의 정책에 대한 관찰 결과(Observation)를 얻고 난 후, 에이전트는 다시 생각 과정을 통해 일본의 정책 정보도 필요하다고 판단합니다. 이에 따라 두 번째 Action으로 japan_ict 도구를 선택하고 "일본의 ICT 주요 정책"을 검색합니다. 이처럼 에이전트는 하나의 복합적인 질문을 해결하기 위해 가용한 도구들을 순차적으로 활용하는 전략을 취합니다.  

두 나라의 정책 정보를 모두 수집한 후, 에이전트는 "미국과 일본의 ICT 주요 정책에 대한 정보를 수집했습니다. 이제 두 국가의 정책의 공통점과 차이점을 분석하겠습니다."라는 생각을 거쳐 공통점과 차이점을 체계적으로 정리한 최종 답변(Final Answer)을 작성하게 됩니다.

In [ ]:
# =========================
# 멀티 쿼리
# =========================
try:
    q = "미국의 ICT 관련 정부 기구, 주요 법령, 국내 기업 진출 사례 각각 따로 검색해. 그렇게 해서 정보 좀 모아봐. 그리고 나서 일본의 AI 정책도 알려줘."
    out = run_react(q, max_iters=10)
    print("최종 답변:", out["output"])
    print("\n=== 실행 로그 (Thought / Action / Action Input / Observation) ===\n")
    print(out["log"])
except Exception as e:
    print(f"멀티 쿼리 중 오류: {e}")

최종 답변: 아래는 요청하신 미국과 일본의 ICT 및 AI 정책 관련 정보입니다.

---

### 1. 미국의 ICT 관련 정부 기구

- **국가통신정보관리청(NTIA, National Telecommunications and Information Administration)**
  - 상무부 산하의 행정기관으로, 통신 및 정보 정책 문제에 대해 대통령에게 자문을 제공.
  - 최근 AI 업무, Open Weight AI 모델 등 AI 관련 정책 권장 사항 개발을 위한 대중 참여 및 의견 요청을 시작.
  - 무선 공급망 혁신 기금 등을 통해 차세대 무선 기술 개발 지원.

- **연방통신위원회(FCC, Federal Communications Commission)**
  - 1934년 연방통신법에 의해 설립된 독립 행정기관.
  - ICT 및 방송에 관한 정책 총괄, 시장 규제, 사업자 허가 및 감독, 주파수 배분 및 할당, 방송편성 등 담당.

---

### 2. 미국의 ICT 관련 주요 법령

- **연방통신법(Communications Act of 1934)**
  - FCC 설립의 근거가 되는 법률로, 미국 내 정보통신 및 방송 산업의 기본 틀을 제공.

- **반도체 과학법(CHIPS and Science Act, 2022)**
  - 미국 내 반도체 산업 경쟁력 강화, 공급망 탄력성 확보, 국가 안보 강화, 주요 기술 접근 지원을 목표로 함.
  - 법률 제정으로 미국 내 반도체·전자 기업에 1,660억 달러 규모의 투자 유치.

- **기타**
  - 미국은 ICT 분야에서 국가 간 기술 교류, 사이버 보안, 6G 연구 등 다양한 법령 및 정책을 추진 중.

---

### 3. 미국 ICT 시장 내 한국 기업 진출 사례

- **한-미 FTA(자유무역협정)**
  - 2012년 발효, 2019년 개정. 투자자-국가분쟁해결제도(ISDS) 및 무역구제 절차 개선.
  - 2022년 FTA 10주년 기념 정부·국회 대표단 미국 방문 등 양국 협

현재 질문은 "미국의 ICT 관련 정부 기구, 주요 법령, 국내 기업 진출 사례, 그리고 일본의 AI 정책"입니다. 질문이 명확하게 네 가지 세부 항목으로 나뉘어 있어서 각각에 대한 개별 검색이 필요한 상황입니다.  

ReAct 에이전트가 시작되자 에이전트는 질문의 의도에 따라서 적절한 도구를 배정하기 위해서 Thought(생각 과정)을 거칩니다. 에이전트는 미국 관련 세 가지 정보를 먼저 retriever_usa로 순차적으로 검색합니다: "미국의 ICT 관련 정부 기구", "미국의 ICT 관련 주요 법령", "미국의 ICT 관련 국내 기업 진출 사례". 각 검색마다 관찰 결과(Observation)를 확인하고 다음 검색으로 넘어갑니다.  

미국 관련 정보 수집을 마친 후, japan_ict 도구를 사용해 "일본의 AI 정책"을 검색합니다. 모든 정보가 수집되면 에이전트는 "이제 최종 답변을 제공할 수 있습니다"라는 생각을 거쳐 체계적인 최종 답변(Final Answer)을 작성합니다.

## 그라디오 데모

- 일본의 AI 정책에 대해 알려줘
- 미국의 ICT 관련 정부 기구는 어떤 것들이 있나요?
- 미국과 일본의 ICT 주요 정책의 공통점과 차이점을 설명해줘
- 이전에 말한 정책들 중에서 가장 중요한 것은 무엇이라고 생각해?

In [ ]:
# ==== Gradio 4.x: Chatbot은 messages 형식을 기대 ====

class ConversationManager:
    def __init__(self):
        self.chat_history = []     # [(user_text, assistant_text), ...] 내부 보관은 유지
        self.execution_logs = []

    def process_message(self, message: str):
        self.execution_logs = []
        try:
            context_str = ""

            # 만약에 이전 대화가 있다면
            if self.chat_history:
                # context_str에다가 이전 대화도 포함한다.
                context_str = "이전 대화 내용:\n"
                for i, (q, a) in enumerate(self.chat_history):
                    context_str += f"질문 {i+1}: {q}\n답변 {i+1}: {a}\n"
                context_str += "\n위 대화 맥락을 고려해서 다음 질문에 답변해주세요.\n\n"

            # full_message에다가 이전 대화를 포함하여 전달.
            full_message = context_str + message

            f = io.StringIO()
            with redirect_stdout(f):
                # 이전 대화도 포함한 프롬프트가 run_react를 동작
                result = run_react(full_message, max_iters=10)
                response = result["output"]
                exec_log = result.get("log", "")

            execution_log = f.getvalue() + "\n" + exec_log
            self.execution_logs.append(execution_log)

            # 내부 기록은 기존대로 유지
            self.chat_history.append((message, response))
            return response, execution_log
        except Exception as e:
            error_message = f"오류가 발생했습니다: {str(e)}"
            self.chat_history.append((message, error_message))
            return error_message, f"실행 중 오류: {str(e)}"

    def clear_history(self):
        self.chat_history = []
        self.execution_logs = []
        return []

In [ ]:
conversation_manager = ConversationManager()

프롬프트 예시
```
다음 질문에 최선을 다해 답변하세요. 당신은 다음 도구들에 접근할 수 있습니다:

japan_ict_trend_searcher: 일본의 ICT 산업의 시장동향 정보를 제공합니다.
usa_ict_trend_searcher: 미국의 ICT 산업의 시장동향 정보를 제공합니다.

다음 형식을 사용하세요:

Question: 답변해야 하는 입력 질문
Thought: 무엇을 할지 항상 생각하세요
Action: 취해야 할 행동, [japan_ict_trend_searcher, usa_ict_trend_searcher, usa_blockchain_trend_searcher] 중 하나여야 합니다. 리스트에 있는 도구 중 1개를 택하십시오.
Action Input: 행동에 대한 입력값
Observation: 행동의 결과
... (이 Thought/Action/Action Input/Observation의 과정이 N번 반복될 수 있습니다)
Thought: 이제 최종 답변을 알겠습니다
Final Answer: 원래 입력된 질문에 대한 최종 답변 (한글로 작성하십시오.)

## 추가적인 주의사항
- 반드시 [Thought -> Action -> Action Input -> Observation] 순서를 준수하십시오. 항상 Action 전에는 Thought가 먼저 나와야 합니다.
- 최종 답변에는 최대한 많은 내용을 포함하십시오.
- 한 번의 검색으로 해결되지 않을 것 같다면 문제를 분할하여 푸는 것도 고려하십시오.
- 정보가 취합되었다면 불필요하게 사이클을 반복하지 마십시오.
- 묻지 않은 정보를 찾으려고 도구를 사용하지 마십시오.

시작하세요!

Question: 이전 대화 내용:
질문 1: 미국의 5G 정책은 어때?
답변 1: 미국의 5G 정책은 크게 세 가지 축으로 구성되어 있습니다. 첫째, 주파수 경매를 통한 스펙트럼 확보입니다. FCC는 28GHz, 24GHz 등 밀리미터파 대역의 경매를 실시하여 통신사들이 5G 네트워크 구축에 필요한 주파수를 확보할 수 있도록 했습니다. 둘째, 인프라 구축 규제 완화입니다. 소형 기지국 설치를 위한 지자체 승인 절차를 간소화하고, 연방 부지 활용을 확대했습니다. 셋째, 국가 보안과 관련된 공급업체 제한 정책입니다. 화웨이, ZTE 등 중국 업체의 장비 사용을 금지하고, 기존 장비의 교체를 위한 지원 프로그램을 운영하고 있습니다.

위 대화 맥락을 고려해서 다음 질문에 답변해주세요.

그러면 일본은 어떤지 알려줘
{agent_scratchpad}
```

In [ ]:
# 위에서 선언한 conversation_manager 객체의 process_message를 호출하는 함수.
# 이 함수는 아래 그라디오 코드에서 msg.submit을 통해 호출됨.
def chat_with_agent(message, history):
    try:
        # conversation_manager.process_message 내부적으로 이전 대화를 포함하여 run_react()를 호출하고 있음.
        response, execution_log = conversation_manager.process_message(message)
    except Exception as e:
        err_msg = f"[오류] {type(e).__name__}: {e}"
        print(err_msg)  # 콘솔에 오류 출력
        response = err_msg
        execution_log = err_msg

    history = (history or []) + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response},
    ]
    return "", history, execution_log

# 위에서 선언한 conversation_manager 객체의 clear_chat_history를 호출하는 함수.
# 이 함수는 아래 그라디오 코드에서 clear.click을 통해 호출됨.
# 즉, 대화를 지우는 버튼을 눌렀을 때 사용됨.
def clear_chat_history():
    conversation_manager.clear_history()
    return [], ""   # 빈 messages, 빈 로그

In [ ]:
with gr.Blocks() as demo:
    gr.HTML("<style>footer{visibility:hidden}</style>")
    gr.Markdown("# ReACT 에이전트")
    gr.Markdown("일본과 미국의 ICT 정책 또는 미국의 블록체인 동향에 관한 질문을 해보세요. 대화 컨텍스트를 유지합니다.")

    with gr.Row():
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=500)  # type 인자 없이 사용. 단, messages 형식 데이터를 넘겨야 함.
            msg = gr.Textbox(label="질문을 입력하세요", placeholder="예: 일본의 AI 정책에 대해 알려줘")
            clear = gr.Button("대화 초기화")

        with gr.Column(scale=1):
            logs = gr.Textbox(label="ReACT 실행 로그", lines=25, max_lines=25)

    # 제가 채팅을 입력해서 엔터키를 누를 때마다 실행되는 게 msg.submit
    msg.submit(chat_with_agent, inputs=[msg, chatbot], outputs=[msg, chatbot, logs])
    clear.click(clear_chat_history, inputs=None, outputs=[chatbot, logs])

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ab3bdabc67c62388b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
